In [ ]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. Load Yamanishi positive DTI dataset
# ==========================================

def load_positive_dti(filepath):
    """
    Expects a CSV with columns: drug, target
    """
    df = pd.read_csv(filepath)
    df = df[['drug', 'target']].drop_duplicates()
    df['pair'] = df['drug'] + "_" + df['target']
    return set(df['pair'].tolist())

positive_pairs = load_positive_dti("yamanishi_positive.csv")
print("Loaded positive pairs:", len(positive_pairs))

# ==========================================
# 2. Load and filter BindingDB
# ==========================================

def convert_to_uM(value, unit):
    """
    Convert BindingDB values to micromolar (µM).
    """
    if pd.isna(value):
        return np.nan
    if unit is None:
        return np.nan
    
    unit = unit.lower()

    # Cases
    if unit in ["um", "µm"]:
        return float(value)
    if unit == "nm":
        return float(value) / 1000.0
    if unit == "pm":
        return float(value) / 1e6
    if unit == "mm":
        return float(value) * 1000
    # Unknown → drop
    return np.nan

def load_bindingdb(filepath):
    df = pd.read_csv(filepath)
    
    # Typical BindingDB columns:
    # "Ligand SMILES", "Ligand Name", "UniProt", "Ki (nM)", "IC50 (nM)", etc.
    
    # Extract drug identifier
    df['drug'] = df['Ligand SMILES']  # use SMILES (authors used OpenBabel later)
    
    # Extract target protein
    if 'UniProt' in df.columns:
        df['target'] = df['UniProt']
    else:
        df['target'] = df['Target Name']
    
    # Consolidate activity columns
    activity_cols = [c for c in df.columns if re.search(r'IC50|Ki|Kd|EC50|Potency', c, re.IGNORECASE)]
    if not activity_cols:
        raise ValueError("Could not find activity columns in BindingDB file.")
    
    # Take first available activity value
    df['activity_raw'] = df[activity_cols].bfill(axis=1).iloc[:,0]
    
    # Extract units from column names (BindingDB often embeds units in col name)
    unit = "nM"
    for c in activity_cols:
        if "(nM)" in c:
            unit = "nM"
        elif "(uM)" in c or "(µM)" in c:
            unit = "uM"
        elif "(mM)" in c:
            unit = "mM"
    
    # Convert to µM
    df['activity_uM'] = df['activity_raw'].apply(lambda x: convert_to_uM(x, unit))
    
    # Filter > 10 µM
    neg = df[df['activity_uM'] > 10][['drug', 'target', 'activity_uM']].dropna()
    
    # Create pair identifier
    neg['pair'] = neg['drug'] + "_" + neg['target']
    
    return neg

neg_bindingdb = load_bindingdb("BindingDB_All.csv")
print("BindingDB negatives (>10 µM):", len(neg_bindingdb))

# ==========================================
# 3. Load and filter BioLiP
# ==========================================

def load_biolip(filepath):
    """
    BioLiP is tab-separated with columns:
    PDB, Chain, UniProt, Ligand, Affinity
    """
    df = pd.read_csv(filepath, sep="\t", header=None)
    
    # BioLiP official column order:
    # 0:PDB, 1:Chain, 2:Resolution, 3:Protein, 4:Ligand, 5:Affinity
    df.columns = ["pdb", "chain", "resolution", "uniprot", "ligand", "affinity"]
    
    df['drug'] = df['ligand']
    df['target'] = df['uniprot']
    
    # Extract numeric affinity
    def parse_affinity(a):
        if isinstance(a, str):
            match = re.findall(r"([\d\.]+)", a)
            if match:
                return float(match[0])
        return np.nan
    
    df['activity_uM'] = df['affinity'].apply(parse_affinity)
    
    # Filter > 10 µM
    neg = df[df['activity_uM'] > 10][['drug', 'target', 'activity_uM']].dropna()
    
    neg['pair'] = neg['drug'] + "_" + neg['target']
    
    return neg

neg_biolip = load_biolip("biolip.txt")
print("BioLiP negatives (>10 µM):", len(neg_biolip))

# ==========================================
# 4. Merge negative sets and remove duplicates
# ==========================================

neg = pd.concat([neg_bindingdb, neg_biolip], ignore_index=True)
neg = neg.drop_duplicates(subset=['pair'])
print("Merged negatives:", len(neg))

# ==========================================
# 5. Remove any negatives that overlap with positives
# ==========================================

neg = neg[~neg['pair'].isin(positive_pairs)]
print("Final negatives after removing positive conflicts:", len(neg))

# ==========================================
# 6. Save final negative dataset
# ==========================================

neg[['drug', 'target', 'activity_uM']].to_csv("negative_dti.csv", index=False)
print("Saved: negative_dti.csv")
